In [0]:
%run ../_shared/00_project_config

In [0]:
%run ../_shared/01_spark_utils

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import StructType


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SOURCE_SYSTEM = "football-data.org"

BRONZE_TABLE = f"{BRONZE_SCHEMA}.football_data_org_standings"
SILVER_TABLE = f"{SILVER_SCHEMA}.football_data_org_standings"

# One row represents one team in a specific competition, season,
# standing type, stage and optional group.
#
# Stage and group may be NULL for some competitions.
# Spark deduplication still partitions NULL values together.

BUSINESS_KEY = [
    "competition_id",
    "season_id",
    "standing_type",
    "stage",
    "standing_group",
    "team_id",
]

REQUIRED_SILVER_COLUMNS = [
    "competition_id",
    "competition_code",
    "league_code",
    "season_id",
    "season_start_year",
    "standing_type",
    "team_id",
    "team_key",
    "position",
    "played_games",
    "points",
]


# -----------------------------------------------------------------------------
# Read Bronze
# -----------------------------------------------------------------------------

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Required Bronze table does not exist: {BRONZE_TABLE}"
    )

bronze_df = spark.table(BRONZE_TABLE)


# -----------------------------------------------------------------------------
# Validate required Bronze columns
# -----------------------------------------------------------------------------

REQUIRED_BRONZE_COLUMNS = [
    "competition",
    "season",
    "type",
    "team",
    "position",
    "playedGames",
    "points",
    "_league",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)


# -----------------------------------------------------------------------------
# Validate required nested fields
# -----------------------------------------------------------------------------

REQUIRED_NESTED_COLUMNS = [
    "competition.id",
    "competition.code",
    "season.id",
    "season.startDate",
    "team.id",
]

missing_nested_columns = [
    path
    for path in REQUIRED_NESTED_COLUMNS
    if not field_exists(bronze_df, path)
]

if missing_nested_columns:
    raise ValueError(
        "Required nested Bronze columns are missing: "
        f"{missing_nested_columns}"
    )


# -----------------------------------------------------------------------------
# Clean source strings
# -----------------------------------------------------------------------------

bronze_df = clean_top_level_strings(bronze_df)


# -----------------------------------------------------------------------------
# Validate source identity
# -----------------------------------------------------------------------------

invalid_source_rows = (
    bronze_df
    .filter(
        F.col("_source_system").isNull()
        | (F.col("_source_system") != F.lit(SOURCE_SYSTEM))
    )
    .limit(1)
    .count()
)

if invalid_source_rows > 0:
    raise ValueError(
        f"Unexpected source system found in {BRONZE_TABLE}."
    )


# -----------------------------------------------------------------------------
# Select, rename and cast Silver columns
# -----------------------------------------------------------------------------

silver_df = (
    bronze_df
    .select(

        # ---------------------------------------------------------------------
        # Competition
        # ---------------------------------------------------------------------

        F.col("competition.id")
        .cast("int")
        .alias("competition_id"),

        clean_string("competition.code")
        .alias("competition_code"),

        optional_string(bronze_df, "competition.name")
        .alias("competition_name"),

        clean_string("_league")
        .alias("league_code"),


        # ---------------------------------------------------------------------
        # Season
        # ---------------------------------------------------------------------

        F.col("season.id")
        .cast("int")
        .alias("season_id"),

        F.col("season.startDate")
        .cast("date")
        .alias("season_start_date"),

        optional_col(bronze_df, "season.endDate", "date")
        .alias("season_end_date"),

        optional_col(bronze_df, "season.currentMatchday", "int")
        .alias("current_matchday"),


        # ---------------------------------------------------------------------
        # Standing classification
        # ---------------------------------------------------------------------

        clean_string("type")
        .alias("standing_type"),

        optional_string(bronze_df, "stage")
        .alias("stage"),

        optional_string(bronze_df, "group")
        .alias("standing_group"),


        # ---------------------------------------------------------------------
        # Team
        # ---------------------------------------------------------------------

        F.col("team.id")
        .cast("int")
        .alias("team_id"),

        optional_string(bronze_df, "team.name")
        .alias("team_name"),

        optional_string(bronze_df, "team.shortName")
        .alias("team_short_name"),

        optional_string(bronze_df, "team.tla")
        .alias("team_tla"),

        optional_string(bronze_df, "team.crest")
        .alias("team_crest"),


        # ---------------------------------------------------------------------
        # Standings position and points
        # ---------------------------------------------------------------------

        F.col("position")
        .cast("int")
        .alias("position"),

        F.col("playedGames")
        .cast("int")
        .alias("played_games"),

        F.col("points")
        .cast("int")
        .alias("points"),


        # ---------------------------------------------------------------------
        # Match results
        # ---------------------------------------------------------------------

        optional_col(bronze_df, "won", "int")
        .alias("won"),

        optional_col(bronze_df, "draw", "int")
        .alias("draw"),

        optional_col(bronze_df, "lost", "int")
        .alias("lost"),


        # ---------------------------------------------------------------------
        # Goals
        # ---------------------------------------------------------------------

        optional_col(bronze_df, "goalsFor", "int")
        .alias("goals_for"),

        optional_col(bronze_df, "goalsAgainst", "int")
        .alias("goals_against"),

        optional_col(bronze_df, "goalDifference", "int")
        .alias("goal_difference"),


        # ---------------------------------------------------------------------
        # Form
        # ---------------------------------------------------------------------

        optional_string(bronze_df, "form")
        .alias("form"),


        # ---------------------------------------------------------------------
        # Bronze lineage
        # ---------------------------------------------------------------------

        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),

        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),

        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Normalize season
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "season_start_year",
    F.year("season_start_date"),
)


# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------

# Stage and standing_group are intentionally excluded here:
# NULL may be valid for these optional dimensions.

validate_required_not_null(
    silver_df,
    [
        "competition_id",
        "season_id",
        "standing_type",
        "team_id",
    ],
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Validate competition-to-league mapping
# -----------------------------------------------------------------------------

expected_league = map_values(
    "competition_code",
    FOOTBALL_DATA_ORG_COMPETITIONS,
)

invalid_mapping_rows = (
    silver_df
    .filter(
        expected_league.isNull()
        | F.col("league_code").isNull()
        | (F.col("league_code") != expected_league)
    )
    .limit(1)
    .count()
)

if invalid_mapping_rows > 0:
    raise ValueError(
        "Competition code and canonical league mapping "
        "are inconsistent."
    )


# -----------------------------------------------------------------------------
# Deduplicate repeated Bronze snapshots
# -----------------------------------------------------------------------------

# Bronze is append-oriented, so the same team's standings may appear
# multiple times after repeated ingestions.
#
# Keep the latest ingested row for each business key.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "_bronze_source_file",
    ],
)



# ---------------------------------------------------------------------------
# Resolve canonical team key
# ---------------------------------------------------------------------------

team_mappings_df = load_team_mappings_by_id(
    spark,
    SILVER_SCHEMA,
    SOURCE_SYSTEM,
)

silver_df = resolve_team_keys_by_id(
    silver_df,
    team_mappings_df,
    source_id_column="team_id",
    output_key_column="team_key",
)

# Ensure standings business keys remain unique.
duplicate_standings = (
    silver_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_standings:
    raise ValueError(
        "Duplicate standings business keys "
        "after canonical team resolution."
    )

print("Canonical standings team resolution completed.")


# -----------------------------------------------------------------------------
# Data quality: validate standing types
# -----------------------------------------------------------------------------

VALID_STANDING_TYPES = [
    "TOTAL",
    "HOME",
    "AWAY",
]

invalid_type_rows = (
    silver_df
    .filter(
        ~F.col("standing_type").isin(VALID_STANDING_TYPES)
    )
    .limit(1)
    .count()
)

if invalid_type_rows > 0:
    raise ValueError(
        "Invalid standing type found. "
        f"Expected one of {VALID_STANDING_TYPES}."
    )


# -----------------------------------------------------------------------------
# Data quality: standings positions must be positive
# -----------------------------------------------------------------------------

invalid_position_rows = (
    silver_df
    .filter(F.col("position") <= 0)
    .limit(1)
    .count()
)

if invalid_position_rows > 0:
    raise ValueError(
        "Invalid standings data: position must be positive."
    )


# -----------------------------------------------------------------------------
# Data quality: statistics must not be negative
# -----------------------------------------------------------------------------

NON_NEGATIVE_COLUMNS = [
    "played_games",
    "won",
    "draw",
    "lost",
    "goals_for",
    "goals_against",
]

negative_value_condition = F.lit(False)

for column_name in NON_NEGATIVE_COLUMNS:
    negative_value_condition = (
        negative_value_condition
        | (F.col(column_name) < 0)
    )

invalid_stat_rows = (
    silver_df
    .filter(negative_value_condition)
    .limit(1)
    .count()
)

if invalid_stat_rows > 0:
    raise ValueError(
        "Invalid standings data: negative statistics found."
    )

# Points are not validated as non-negative because some competitions
# can apply deductions, potentially producing negative points.
#
# Goal difference can also legitimately be negative.


# -----------------------------------------------------------------------------
# Data quality: match results consistency
# -----------------------------------------------------------------------------

# When all result components are available, their sum should equal
# the number of played games.

invalid_results_rows = (
    silver_df
    .filter(
        F.col("won").isNotNull()
        & F.col("draw").isNotNull()
        & F.col("lost").isNotNull()
        & (
            F.col("played_games")
            != (
                F.col("won")
                + F.col("draw")
                + F.col("lost")
            )
        )
    )
    .limit(1)
    .count()
)

if invalid_results_rows > 0:
    raise ValueError(
        "Invalid standings data: "
        "won + draw + lost does not equal played_games."
    )


# -----------------------------------------------------------------------------
# Data quality: goal difference consistency
# -----------------------------------------------------------------------------

invalid_goal_difference_rows = (
    silver_df
    .filter(
        F.col("goals_for").isNotNull()
        & F.col("goals_against").isNotNull()
        & F.col("goal_difference").isNotNull()
        & (
            F.col("goal_difference")
            != (
                F.col("goals_for")
                - F.col("goals_against")
            )
        )
    )
    .limit(1)
    .count()
)

if invalid_goal_difference_rows > 0:
    raise ValueError(
        "Invalid standings data: "
        "goal_difference does not equal goals_for - goals_against."
    )


# -----------------------------------------------------------------------------
# Data quality: unique positions within standings tables
# -----------------------------------------------------------------------------

# Two different teams should not have the same position in the
# same standings table.

STANDING_POSITION_KEY = [
    "competition_id",
    "season_id",
    "standing_type",
    "stage",
    "standing_group",
    "position",
]

duplicate_positions = (
    silver_df
    .groupBy(*STANDING_POSITION_KEY)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_positions > 0:
    raise ValueError(
        "Duplicate standings positions found "
        "within the same competition, season, type, stage and group."
    )


# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final validation
# -----------------------------------------------------------------------------

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)

if validation_metrics["row_count"] == 0:
    raise ValueError(
        "Silver standings dataset is empty. "
        "Refusing to overwrite the target table."
    )


# -----------------------------------------------------------------------------
# Write Silver
# -----------------------------------------------------------------------------

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"standings records into {SILVER_TABLE}."
)
